# ShopSmart Food Catalog — project starter

**Purpose:** obtain the first *comparable* estimates, inspect actual cases, and leave with a defensible project question.

Read `datasheet.md` and `system_card.md` before interpreting a number. This notebook uses only the `analysis` split. The course-designated `holdout` remains sealed.


## 1. Check the package and lock the common evaluation cases

The floor, fitted baseline, and AI system will all be scored on the same internal evaluation cases. The baseline is trained on the remaining analysis cases.

The first code cell finds this track's folder or downloads its zip from the course Box folder, checks that the download matches the course copy (SHA-256 hash), and unpacks it next to this notebook. If the course link cannot be downloaded directly, the cell tells you how to upload the zip yourself; the same check runs either way.


In [ ]:
from pathlib import Path
import hashlib
import json
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

TRACK_ZIP = {'file': 'track_1_food.zip', 'bytes': 22870667, 'sha256': 'ca07ce4745515f2ec24d6757272bd6ba75a560610bf9a501af3a96040f1e04c3', 'url': 'https://cmu.box.com/shared/static/8p9z8srczc6kcewed7le1u19sgwboyp1.zip'}

def get_track_package(zip_info):
    """Use this track's folder if it is here; otherwise download, check, and unpack its zip."""
    name = zip_info["file"].removesuffix(".zip")
    here = Path.cwd()
    for folder in [here, here / name]:
        if folder.name == name and (folder / "metadata.csv").exists():
            return folder
    archive = here / zip_info["file"]
    upload_by_hand = (f"Download {zip_info['file']} from the course Box folder yourself, upload it to "
                      f"{here} (in Colab, use the Files panel), and rerun this cell.")
    if not archive.exists():
        if not zip_info["url"]:
            raise FileNotFoundError(upload_by_hand)
        print(f"Downloading {zip_info['file']} ({zip_info['bytes'] / 1e6:.0f} MB) ...")
        partial = here / (zip_info["file"] + ".part")
        try:
            urllib.request.urlretrieve(zip_info["url"], partial)
        except OSError as error:
            partial.unlink(missing_ok=True)
            raise FileNotFoundError(f"The download did not work ({error}). {upload_by_hand}") from None
        if hashlib.sha256(partial.read_bytes()).hexdigest() != zip_info["sha256"]:
            partial.unlink()  # often a Box sign-in page rather than the zip
            raise FileNotFoundError(f"The download is not the course zip. {upload_by_hand}")
        partial.replace(archive)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != zip_info["sha256"]:
        raise ValueError(f"{archive.name} does not match the course copy. Delete it and "
                         "download it from the course Box folder again.")
    with zipfile.ZipFile(archive) as z:
        z.extractall(here)
    print(f"Unpacked {archive.name} into {name}/.")
    return here / name

ROOT = get_track_package(TRACK_ZIP)

def wilson_interval(correct, n, z=1.96):
    """Wilson interval for a binomial proportion; returns (low, high)."""
    if n == 0:
        return (np.nan, np.nan)
    p = correct / n
    den = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / den
    half = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / den
    return (center - half, center + half)

def paired_bootstrap_delta(ai_correct, baseline_correct, B=4000, seed=2027):
    """Percentile interval for mean(AI correct - baseline correct), paired by case."""
    ai = np.asarray(ai_correct, dtype=float)
    base = np.asarray(baseline_correct, dtype=float)
    if len(ai) != len(base) or len(ai) == 0:
        raise ValueError("Paired comparison requires the same nonempty cases.")
    rng = np.random.default_rng(seed)
    draws = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, len(ai), len(ai))
        draws[b] = np.mean(ai[idx] - base[idx])
    return float(np.mean(ai-base)), tuple(np.quantile(draws, [0.025, 0.975]))

def result_row(system, correct):
    correct = np.asarray(correct, dtype=bool)
    lo, hi = wilson_interval(int(correct.sum()), len(correct))
    return {
        "system": system,
        "correct": int(correct.sum()),
        "n": int(len(correct)),
        "estimate": correct.mean(),
        "ci_low": lo,
        "ci_high": hi,
    }

def select_cases(cases, n, id_col, confidence_col, seed=2027):
    """Lock the selection rule before anything is displayed.

    Among cases where the baseline and AI differ in correctness, draw up to n/2
    where the AI is wrong and n/2 where the baseline is wrong (seeded draw).
    """
    pool = cases.loc[cases["baseline_correct"] != cases["ai_correct"]]
    if pool.empty:
        pool = cases
    parts = [group.sample(min(len(group), n // 2), random_state=seed)
             for _, group in pool.groupby("ai_correct")]
    return pd.concat(parts).sort_values(["ai_correct", confidence_col, id_col], ascending=[True, False, True])

print("Package folder:", ROOT.name)


from sklearn.linear_model import LogisticRegression

meta = pd.read_csv(ROOT / "metadata.csv")
splits = pd.read_csv(ROOT / "splits.csv")
ai = pd.read_csv(ROOT / "ai_outputs" / "clip_predictions.csv")
features = np.load(ROOT / "data" / "baseline_features.npy")
feature_ids = np.load(ROOT / "data" / "feature_row_labels.npy", allow_pickle=True).astype(str)

assert meta.item_id.is_unique and splits.item_id.is_unique and ai.item_id.is_unique
assert set(meta.item_id) == set(splits.item_id) == set(ai.item_id) == set(feature_ids)
assert features.shape[0] == len(meta) == len(feature_ids)

data = meta.merge(splits, on="item_id", validate="one_to_one").merge(
    ai, on=["item_id", "true_label"], validate="one_to_one"
)
analysis = data.loc[data.split == "analysis"].copy()
holdout_n = int((data.split == "holdout").sum())
print(f"Analysis units: {len(analysis):,}; sealed holdout units: {holdout_n:,}")

train_ids, eval_ids = train_test_split(
    analysis.item_id,
    test_size=.25,
    random_state=2027,
    stratify=analysis.true_label,
)
train_ids, eval_ids = set(train_ids), set(eval_ids)
print(f"Internal training units: {len(train_ids):,}; common evaluation units: {len(eval_ids):,}")


## 2. Fit the statistical baseline and compare it fairly

The majority floor is learned from the internal training cases. The fitted baseline is multinomial logistic regression on the supplied, documented image-summary features. It does not use CLIP scores or holdout labels.


In [ ]:
row_for_id = {item_id: i for i, item_id in enumerate(feature_ids)}
train = analysis.loc[analysis.item_id.isin(train_ids)].copy()
evaluation = analysis.loc[analysis.item_id.isin(eval_ids)].copy()

X_train = features[[row_for_id[x] for x in train.item_id]]
X_eval = features[[row_for_id[x] for x in evaluation.item_id]]
y_train = train.true_label.to_numpy()
y_eval = evaluation.true_label.to_numpy()

majority_label = train.true_label.value_counts().index[0]
floor_pred = np.repeat(majority_label, len(evaluation))

baseline = LogisticRegression(max_iter=2000, solver="lbfgs")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_eval)
ai_pred = evaluation.pred_label.to_numpy()

floor_correct = floor_pred == y_eval
baseline_correct = baseline_pred == y_eval
ai_correct = ai_pred == y_eval

results = pd.DataFrame([
    result_row(f"majority floor ({majority_label})", floor_correct),
    result_row("multinomial logistic baseline", baseline_correct),
    result_row("supplied CLIP system", ai_correct),
])
display(results.style.format({"estimate":"{:.3f}", "ci_low":"{:.3f}", "ci_high":"{:.3f}"}))

delta, delta_ci = paired_bootstrap_delta(ai_correct, baseline_correct)
print(f"Paired AI - baseline accuracy: {delta:+.3f} "
      f"(paired bootstrap 95% interval {delta_ci[0]:+.3f}, {delta_ci[1]:+.3f}; n={len(y_eval)})")


### Track-specific check

State one scientifically meaningful alternative label rule or operating decision before computing it.


## 3. Check the cached AI outputs

Cached predictions are measurements produced by a fixed system, prompt, class set, and preprocessing pipeline. Coverage and score validity are part of checking the data.


In [ ]:
pcols = [c for c in ai.columns if c.startswith("p_")]
checks = pd.Series({
    "metadata rows": len(meta),
    "cached-output rows": len(ai),
    "duplicate item IDs": int(ai.item_id.duplicated().sum()),
    "missing cached rows": int((~meta.item_id.isin(ai.item_id)).sum()),
    "probability columns": len(pcols),
    "rows whose scores do not sum to 1 (tolerance .001)": int((~np.isclose(ai[pcols].sum(axis=1), 1, atol=.001)).sum()),
    "rows where cached pred is not score argmax": int((
        ai[pcols].idxmax(axis=1).str.removeprefix("p_").to_numpy() != ai.pred_label.to_numpy()
    ).sum()),
})
display(checks.to_frame("value"))


## 4. Browse selected cases

Selection rule: among common evaluation cases where the fitted baseline and AI system differ in correctness, draw three where the AI is wrong and three where the baseline is wrong (seed 2027). The rule is applied before images appear.


In [ ]:
evaluation = evaluation.reset_index(drop=True)
pcols = [c for c in evaluation.columns if c.startswith("p_")]
comparison_cases = evaluation[["item_id", "true_label", "image_path"]].copy()
comparison_cases["baseline_pred"] = baseline_pred
comparison_cases["ai_pred"] = ai_pred
comparison_cases["baseline_correct"] = baseline_correct
comparison_cases["ai_correct"] = ai_correct
comparison_cases["ai_confidence"] = evaluation[pcols].max(axis=1).to_numpy()


In [ ]:
def browse_image_cases(cases, n=6):
    """Select cases by the locked rule, then display them."""
    chosen = select_cases(cases, n, "item_id", "ai_confidence")  # complete before anything is displayed
    paths = [ROOT / p for p in chosen["image_path"]]

    if not any(p.exists() for p in paths):
        print("The photos are missing from data/images/. Delete this track's folder and zip, rerun "
              "the first code cell to unpack the course zip again, and then rerun this cell. "
              "The selected cases are listed below.")
        display(chosen[["item_id", "true_label", "baseline_pred", "ai_pred", "ai_confidence"]])
        return chosen

    from PIL import Image
    cols = 3
    rows = int(np.ceil(len(chosen) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4*rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, row), path in zip(axes, chosen.iterrows(), paths):
        if path.exists():
            ax.imshow(Image.open(path).convert("RGB"))
        else:
            ax.text(.5, .5, "image missing", ha="center", va="center")
        ax.set_title(
            f"truth: {row.true_label}\nbaseline: {row.baseline_pred} | AI: {row.ai_pred}\n"
            f"AI score: {row.ai_confidence:.2f}"
        )
        ax.axis("off")
    for ax in axes[len(chosen):]:
        ax.axis("off")
    plt.tight_layout()
    return chosen

selected_cases = browse_image_cases(comparison_cases)


**Case note (complete before more modeling):** What visible pattern, ambiguity, or data-quality issue appears in the selected cases? How could it change the estimand, subgroup plan, or sensitivity analysis?

> Write 3–5 sentences here.


## 5. Who is your client? Reweight to the client's mix

Lecture 9A showed that a benchmark's accuracy is a weighted average of the accuracies within groups, weighted by the benchmark's own mix of groups. Your client's mix is probably different. This section finds the AI system's accuracy in each group, then reweights it to a mix you choose for your client.

It uses **all analysis cases**, because the AI system was not fitted on any of them. The comparison with the baseline in section 2 uses only the evaluation cases, so the overall numbers can differ a little. The holdout stays sealed.


In [ ]:
def accuracy_by_group(groups, correct):
    """One row per group: cases, the benchmark's share, and accuracy with a Wilson interval."""
    table = pd.DataFrame({"group": groups, "correct": correct})
    total = len(table)
    rows = []
    for group_name, cases in table.groupby("group"):
        n = len(cases)                       # cases in this group
        right = int(cases["correct"].sum())  # cases the AI system got right
        low, high = wilson_interval(right, n)
        rows.append({"group": group_name, "cases": n, "benchmark_share": n / total,
                     "accuracy": right / n, "ci_low": low, "ci_high": high})
    return pd.DataFrame(rows).sort_values("accuracy").reset_index(drop=True)


def plot_by_group(by_group, label):
    """Accuracy in each group with its 95% Wilson interval, lowest first."""
    fig, ax = plt.subplots(figsize=(8, 0.3 * len(by_group) + 1.2))
    for i, row in by_group.iterrows():
        ax.errorbar(row["accuracy"], i,
                    xerr=[[row["accuracy"] - row["ci_low"]], [row["ci_high"] - row["accuracy"]]],
                    fmt="o", color="#155a8a", capsize=2)
    ax.set_yticks(range(len(by_group)))
    ax.set_yticklabels(by_group["group"])
    ax.set_xlim(0, 1)
    ax.set_xlabel(f"AI system's accuracy within each {label}, with 95% Wilson interval")
    plt.tight_layout()
    plt.show()


def reweight(by_group, shares):
    """Accuracy for a mix of groups, its standard error, and Kish's effective sample size."""
    total_share = sum(shares.values())
    estimate = 0.0
    variance = 0.0
    weight_sum = 0.0
    for _, row in by_group.iterrows():
        share = shares.get(row["group"], 0.0) / total_share  # shares are rescaled to add to 1
        p = row["accuracy"]
        n = row["cases"]
        estimate = estimate + share * p
        variance = variance + share**2 * p * (1 - p) / n
        weight_sum = weight_sum + share**2 / n
    return estimate, np.sqrt(variance), 1 / weight_sum


In [ ]:
# Groups: the dish in the photo (the right answer).
# The AI system is right when CLIP's prediction matches the label.
groups = analysis["true_label"].to_numpy()
all_correct = (analysis["pred_label"] == analysis["true_label"]).to_numpy()
GROUP_LABEL = 'dish'

by_group = accuracy_by_group(groups, all_correct)
display(by_group.style.format({"benchmark_share": "{:.3f}", "accuracy": "{:.3f}",
                               "ci_low": "{:.3f}", "ci_high": "{:.3f}"}))
plot_by_group(by_group, GROUP_LABEL)


**Reading the table.** Grouping by dish is grouping by the right answer, like the empty and animal photos in Lecture 9A. Reweighting assumes CLIP does as well on your client's photos of each dish as on these. Food-101 records no photographer or listing, so the photos are treated as independent.


In [ ]:
# The benchmark's own mix of groups
benchmark_shares = {}
for _, row in by_group.iterrows():
    benchmark_shares[row["group"]] = row["benchmark_share"]

# Your client's mix: start from a copy of the benchmark's mix, then change it below.
client_shares = dict(benchmark_shares)

# ---- Edit here. Shares are rescaled to add to 1, so counts work too. ----
# client_shares["name of a group"] = 0.40
# A group that is not in your client's population can be set to 0.

benchmark_estimate, benchmark_se, benchmark_neff = reweight(by_group, benchmark_shares)
client_estimate, client_se, client_neff = reweight(by_group, client_shares)

print(f"Benchmark's mix: {benchmark_estimate:.3f} "
      f"(95% interval {benchmark_estimate - 1.96 * benchmark_se:.3f} to "
      f"{benchmark_estimate + 1.96 * benchmark_se:.3f}), from {by_group['cases'].sum():,} cases")
print(f"Your client's mix: {client_estimate:.3f} "
      f"(95% interval {client_estimate - 1.96 * client_se:.3f} to "
      f"{client_estimate + 1.96 * client_se:.3f}), worth about {client_neff:.0f} equally weighted cases")
print(f"Client minus benchmark: {client_estimate - benchmark_estimate:+.3f}")


**Where your client's mix comes from (one or two sentences).** A published figure, the client's own records, or a reasoned guess? Say which, and why it is plausible.

> Write it here.

**Population paragraph (draft it before you leave; it becomes "who the numbers are about" in Milestone 1).** In four or five sentences: name the population the benchmark describes and the one your client cares about; say how they differ, using the table above; give the reweighted accuracy with its interval and the assumption it rests on; say whether the benchmark's number is likely too high or too low for your client, or why the direction is unknown; and name the data that would check it.

> Write it here.


## 6. Project-launch handoff: start the Milestone 1 outline

Fill in what you can before you leave the launch session. The five headings below are the five parts of Milestone 1; Milestone 1 in the Project Description says what goes in each.

**The problem.**

>

**Who the numbers are about.** Start from your section 5 paragraph.

>

**EDA.** Use what you found in sections 1 to 5.

>

**Ideas for the analysis.**

>

**Open questions.**

>
